In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

# File Import

In [ ]:
df=pd.read_csv('lung_disease.csv')

In [ ]:
df.head()

In [ ]:
df['smoke'].replace({'T': 1, 'F': 0},inplace=True)
df['PEFR'].replace({'F': 1, 'T': 0},inplace=True)
df['O2'].replace({'F': 1, 'T': 0},inplace=True)
df['PEFR'].replace({'F': 1, 'T': 0},inplace=True)
df[ 'ABG-P-O2'].replace({'F': 1, 'T': 0},inplace=True)
df[ 'ABG-P-CO2'].replace({'F': 1, 'T': 0},inplace=True)
df[ 'ABG-pH Level'].replace({'F': 1, 'T': 0},inplace=True)
df[ 'Scan'].replace({'MRI':1, 'X-ray':2 ,'CT':3},inplace=True)
df[ 'Asthama'].replace({'F': 0, 'T':1 },inplace=True)
df[ 'Other diseaes'].replace({'F': 0, 'T':1 },inplace=True)
df[ 'Risk'].replace({'F':0, 'T':1 },inplace=True)
df.head()

# Data Labeling

In [ ]:
#Input Data
x=df.iloc[:,1:-1].values

In [ ]:
x

In [ ]:
# Output 
y=df.iloc[:,-1].values

In [ ]:
y

# Training Data

In [ ]:
from sklearn.model_selection import train_test_split

Train Test Split

In [ ]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.3,random_state=1)

Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
sc=StandardScaler()
xtrain=sc.fit_transform(xtrain)
xtest=sc.transform(xtest)

In [ ]:
xtrain

# Training Neural Network

In [ ]:
from tensorflow.keras import layers, models

In [ ]:
#initialize model
ann=Sequential()
#add layers
ann.add(Dense(units=13,activation="relu"))#Input Layer
ann.add(Dense(units=5,activation="relu")) #hidden layer
ann.add(Dense(units=1,activation="sigmoid"))#output Layer


In [ ]:
ann.compile(optimizer="adam",loss="binary_crossentropy")

In [ ]:
#compile
ann.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
#train
ann.fit(xtrain,ytrain,batch_size=30,epochs=500)

In [ ]:
ann.history.history

# Loss vs Accuracy Graph

In [ ]:
lossgraph=pd.DataFrame(ann.history.history)
lossgraph.plot()

In [ ]:
ypred=ann.predict(xtest)
ypred

In [ ]:
ypred=ann.predict(xtest)
ypred=ypred>0.5

# Classification Report

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(ytest,ypred))

# Regularization

Early Stopping

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
earlystop=EarlyStopping(monitor="val_loss",mode="min",verbose=1,patience=250)
#train the model
ann.fit(xtrain,ytrain,epochs=500,validation_data=(xtest,ytest),callbacks=[earlystop])

Epoch 252/350
11/11 [==============================] - 0s 3ms/step - loss: 0.1297 - accuracy: 0.9574 - val_loss: 0.9054 - val_accuracy: 0.8156
Epoch 252: early stopping

In [ ]:
ES=pd.DataFrame(ann.history.history)
ES.plot()

DropOut


In [ ]:
from tensorflow.keras.layers import Dropout

In [ ]:
ann=Sequential()
ann.add(Dense(units=13,activation="relu"))
ann.add(Dropout(rate=0.5))
ann.add(Dense(units=5,activation="relu"))
ann.add(Dropout(rate=0.5))
ann.add(Dense(units=5,activation="relu"))
ann.add(Dropout(rate=0.5))
ann.add(Dense(units=1,activation='sigmoid'))
ann.compile(optimizer="adam",loss="binary_crossentropy")

In [ ]:
earlystop=EarlyStopping(monitor="val_loss",mode="min",verbose=1,patience=25)
#train the model
ann.fit(xtrain,ytrain,epochs=350,validation_data=(xtest,ytest),callbacks=[earlystop])

In [ ]:
lossdf=pd.DataFrame(ann.history.history)
lossdf.plot()

In [ ]:
ann.evaluate(xtrain,ytrain)

In [ ]:
ann.evaluate(xtest,ytest)

In [ ]:
ypred=ann.predict(xtest)

# Predicting New Unknown Value

In [ ]:
#unknown data prediction
newdata=[[1,2.85,2.16,1,1,1,0,0,2,0,0,60]]
newdata=sc.transform(newdata)
ann.predict(newdata)

# Making Model from the Neural Network

In [ ]:
from tensorflow.keras.models import load_model

In [ ]:
ann.save("Lungs Disease Model")

In [ ]:
model=load_model("Lungs Disease")

In [ ]:
model.predict(newdata)